# 02 — Train the confidence/quality head

Ports `scripts/train_confidence_head_standalone.py` (already verified working
in development — see `docs/phase3_confidence_head_training.md` for the full
writeup and honest results) to run on Kaggle, optionally at a larger scale
than the 200-image local run.

**This does NOT need detectron2 or the Swin-L backbone at all** — much
simpler setup than notebooks 00/01. `ConfidenceHead` is designed to eventually
sit on the real detector's features, but that backbone only means anything
once trained on real caries data (notebook 01), which takes a long time. This
notebook validates the confidence head's actual task — can it learn to
predict degradation type/severity from an image — against a small stand-in
CNN trunk instead, so progress on it doesn't have to wait for 01 to finish.
When 01 produces a real trained checkpoint, retrain this head against its
real FPN p5 features instead of the stand-in trunk here.

Can run entirely on CPU (development run took well under a minute for 200
images); GPU just lets you scale up the image count and epoch count if you
want a stronger proof of concept.

## 1. Setup — no detectron2 needed

In [ ]:
!git clone https://github.com/christopherh-88/Carries-Confidence.git
%cd Carries-Confidence
!pip install -q -r requirements-core.txt
!pip install -q torch  # if not already present in this Kaggle image

import torch
print("CUDA available:", torch.cuda.is_available())

## 2. Dataset

Same DENTEX path as notebooks 00/01.

In [ ]:
import os
DATA_ROOT = "/kaggle/input/dentex/DENTEX/training_data/quadrant-enumeration-disease"
assert os.path.exists(f"{DATA_ROOT}/xrays"), f"expected images at {DATA_ROOT}/xrays -- check DATA_ROOT"
print("DATA_ROOT ok:", DATA_ROOT)

## 3. Train

Same trunk + `ConfidenceHead` + training loop as the verified standalone
script, with `N_TRAIN_IMAGES`/`N_VAL_IMAGES` bumped up since Kaggle has more
compute available than a laptop — tune these based on your session's time
budget. All 705 images are available; the local dev run only used 200/50.

In [ ]:
import sys
sys.path.insert(0, ".")
import random
import cv2
import numpy as np
import torch
import torch.nn as nn

from src.data.degradation import DEGRADATION_NAMES, apply_degradations
from src.data.dentex import load_coco, patient_level_split
from src.models.confidence_head import ConfidenceHead
from src.utils.seed import set_seed

IMAGE_ROOT = f"{DATA_ROOT}/xrays"
COCO_JSON = f"{DATA_ROOT}/train_quadrant_enumeration_disease.json"
IMG_SIZE = 256
VARIANTS_PER_IMAGE = 4
N_TRAIN_IMAGES = 500   # up from the local dev run's 200 -- tune for your time budget
N_VAL_IMAGES = 100     # up from 50
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class TinyTrunk(nn.Module):
    """Small stand-in CNN, NOT the real detector backbone -- see the note
    at the top of this notebook for why that's the right call for now."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, stride=2, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)


def load_image(file_name):
    img = cv2.imread(os.path.join(IMAGE_ROOT, file_name), cv2.IMREAD_COLOR)
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE))


def build_examples(file_names, seed):
    rng = random.Random(seed)
    examples = []
    for fn in file_names:
        base = load_image(fn)
        examples.append((base, np.zeros(len(DEGRADATION_NAMES), dtype=np.float32)))
        for _ in range(VARIANTS_PER_IMAGE - 1):
            result = apply_degradations(base, seed=rng.randint(0, 1_000_000))
            examples.append((result.image, result.label_vector()))
    return examples


def to_tensor_batch(examples):
    imgs = np.stack([e[0] for e in examples]).astype(np.float32) / 255.0
    imgs = torch.from_numpy(imgs).permute(0, 3, 1, 2)
    labels = torch.from_numpy(np.stack([e[1] for e in examples]))
    return imgs, labels


set_seed(0)
coco = load_coco(COCO_JSON)
split = patient_level_split(coco, val_frac=0.15, test_frac=0.15, seed=0)
id_to_name = {im["id"]: im["file_name"] for im in coco["images"]}

train_files = [id_to_name[i] for i in split["train"][:N_TRAIN_IMAGES]]
val_files = [id_to_name[i] for i in split["val"][:N_VAL_IMAGES]]
print(f"train images: {len(train_files)}, val images: {len(val_files)}")

train_examples = build_examples(train_files, seed=1)
val_examples = build_examples(val_files, seed=2)
train_x, train_y = to_tensor_batch(train_examples)
val_x, val_y = to_tensor_batch(val_examples)
train_x, train_y = train_x.to(DEVICE), train_y.to(DEVICE)
val_x, val_y = val_x.to(DEVICE), val_y.to(DEVICE)
train_usability_target = 1.0 - train_y.max(dim=1).values
val_usability_target = 1.0 - val_y.max(dim=1).values
print(f"train examples: {len(train_examples)}, val examples: {len(val_examples)}")

trunk = TinyTrunk().to(DEVICE)
head = ConfidenceHead(in_features=256).to(DEVICE)
optimizer = torch.optim.Adam(list(trunk.parameters()) + list(head.parameters()), lr=1e-3)
loss_fn = nn.SmoothL1Loss()

n_epochs = 15
batch_size = 16
n_train = train_x.shape[0]

for epoch in range(n_epochs):
    trunk.train(); head.train()
    perm = torch.randperm(n_train)
    epoch_loss = 0.0
    for i in range(0, n_train, batch_size):
        idx = perm[i:i + batch_size]
        optimizer.zero_grad()
        feats = trunk(train_x[idx])
        severity_pred, usability_pred = head(feats)
        loss = loss_fn(severity_pred, train_y[idx]) + loss_fn(usability_pred, train_usability_target[idx])
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * idx.shape[0]
    epoch_loss /= n_train

    trunk.eval(); head.eval()
    with torch.no_grad():
        val_severity_pred, val_usability_pred = head(trunk(val_x))
        val_loss = (loss_fn(val_severity_pred, val_y) + loss_fn(val_usability_pred, val_usability_target)).item()
    print(f"epoch {epoch+1:2d}/{n_epochs}  train_loss={epoch_loss:.4f}  val_loss={val_loss:.4f}")


## 4. Full validation-set summary — the honest numbers

The development run (200/50 images) got: 69.3% dominant-degradation accuracy
(vs 20% chance), 0.593 usability correlation, and a measurable overconfidence
bias on degraded images (0.696 predicted vs 0.321 true mean usability). See
if a larger run here does better, and whether the overconfidence bias
persists — if it does, that's a real finding to carry into how `decide()`'s
thresholds eventually get tuned.

In [ ]:
nonclean_mask = val_y.max(dim=1).values > 0
true_dom = val_y[nonclean_mask].argmax(dim=1)
pred_dom = val_severity_pred[nonclean_mask].argmax(dim=1)
dom_acc = (true_dom == pred_dom).float().mean().item()
chance = 1.0 / len(DEGRADATION_NAMES)

corr = np.corrcoef(val_usability_pred.cpu().numpy(), val_usability_target.cpu().numpy())[0, 1]
clean_mean = val_usability_pred[~nonclean_mask].mean().item()
degraded_mean = val_usability_pred[nonclean_mask].mean().item()
degraded_true_mean = val_usability_target[nonclean_mask].mean().item()

print(f"dominant-degradation accuracy: {dom_acc:.3f}  (n={int(nonclean_mask.sum())}, chance={chance:.3f})")
print(f"usability score Pearson correlation (pred vs true): {corr:.3f}")
print(f"mean predicted usability -- CLEAN images: {clean_mean:.3f} (true=1.0)")
print(f"mean predicted usability -- DEGRADED images: {degraded_mean:.3f} (true mean={degraded_true_mean:.3f})")
print()
print("compare these to the local dev run's numbers in docs/phase3_confidence_head_training.md")